Merge all samples, normalize jointly, integrate with Harmony, cluster with Leiden and annotate brain regions.
- Leiden results differ between machines, so the annotation refers to the saved clusters in `adatas/20220923_intermediate_umap_save_integrated.h5ad`.
- Its annotated version, `adatas/20240104_integrated_annotated.h5ad`, is used from here on.

In [ ]:
from pathlib import Path
import anndata as ad
import harmonypy
import matplotlib.pyplot as plt
import scanpy as sc
from matplotlib.pyplot import rc_context
base = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/FOXF2/Visium")

In [ ]:
order = ["735", "738", "740", "733", "739", "737", "741", "734"]  # original concat order, keeps obs_names
merged = ad.concat({s: sc.read_h5ad(base / "adatas" / "per_sample" / f"{s}.h5ad") for s in order}, join="outer", merge="first")
merged.var = merged.var[["gene_ids", "feature_types", "genome", "mt"]]
merged.obs_names_make_unique()

merged.layers["normalized_jointly"] = merged.layers["counts"].copy()
sc.pp.normalize_total(merged, layer="normalized_jointly")
sc.pp.log1p(merged, layer="normalized_jointly")
merged.X = merged.layers["normalized_jointly"]

sc.pp.highly_variable_genes(merged, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.tl.pca(merged, n_comps=40)
z = harmonypy.run_harmony(merged.obsm["X_pca"], merged.obs, "sample").Z_corr
merged.obsm["X_pca_harmony"] = z if z.shape[0] == merged.n_obs else z.T  # orientation differs between harmonypy versions
sc.pp.neighbors(merged, use_rep="X_pca_harmony")
sc.tl.umap(merged)
sc.tl.leiden(merged, key_added="clusters", resolution=6.5)
with rc_context({"figure.figsize": (9, 9)}):
    sc.pl.umap(merged, color=["clusters", "sample", "condition"], wspace=0.6)

Annotation, based on spatial location of clusters

In [ ]:
merged = sc.read_h5ad(base / "adatas" / "20220923_intermediate_umap_save_integrated.h5ad")

In [ ]:
regions = [
    "Thalamus1", "Hypothalamus", "HippocampusCA", "Cortex3", "Subcortical_Mixed", "Cortex2", "Striatum", "Thalamus2", "Cortex2", "Striatum",
    "HippocampusCA", "StriatumAmygdala", "Cortex2", "FiberTract", "Cortex3", "Hypothalamus", "Cortex4", "Thalamus1", "StriatumAmygdala", "Cortex3",
    "Subcortical_Mixed", "Cortex4", "Cortex3", "Cortex4", "ThalamusRT", "Cortex4", "Cortex4", "Hypothalamus", "Thalamus1", "FiberTract",
    "Cortex1", "Cortex2", "Cortex3", "StriatumAmygdala", "Cortex3", "Thalamus1", "Cortex3", "Hypothalamus", "HippocampusDG", "FiberTract",
    "Thalamus1", "FiberTract", "HippocampusCA", "Thalamus1", "Cortex3", "Striatum", "Cortex1", "FiberTract", "HippocampusCA", "FiberTract",
    "Cortex2", "Striatum", "Cortex4", "Ventricle", "Cortex2", "FiberTract", "Epithalamus", "Epithalamus", "Hypothalamus", "Thalamus2",
]
broad = {"Cortex1": "Cortex", "Cortex2": "Cortex", "Cortex3": "Cortex", "Cortex4": "Cortex", "HippocampusCA": "Hippocampus", "HippocampusDG": "Hippocampus",
         "Thalamus1": "Thalamus", "Thalamus2": "Thalamus", "ThalamusRT": "Thalamus", "Epithalamus": "Thalamus"}
merged.obs["clusters_new"] = merged.obs["clusters"].map(dict(zip(map(str, range(len(regions))), regions))).astype("category")
merged.obs["clusters_broad"] = merged.obs["clusters_new"].astype(str).replace(broad).astype("category")
with rc_context({"figure.figsize": (10, 10)}):
    sc.pl.umap(merged, color=["clusters_broad", "clusters_new"], wspace=0.25, size=15)

In [ ]:
for s in order:
    sub = merged[merged.obs["sample"] == s]
    sc.pl.spatial(sub, color=["clusters_new", "clusters_broad"], spot_size=97, size=1.4, frameon=False, wspace=0.4, show=False)
    plt.savefig(base / "plots" / f"spatial_regions_{s}.svg", bbox_inches="tight")
    plt.show()

In [ ]:
merged.write_h5ad(base / "adatas" / "integrated_annotated.h5ad", compression="gzip")